# Anatomy → Anatomy Relation Pipeline

Builds a unified, deduplicated edge table for the **Anatomy–Anatomy** relation  
by ingesting processed files from multiple KG sources, normalising identifiers  
if needed, and writing the final triple table to disk.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch
- Pheknowlator
- PrimeKG
- TAR-KG

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [18]:
import pandas as pd

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/ANATOMY_ANATOMY/ALL_ANATOMY_ANATOMY.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1 · Load Source Files

Each source DataFrame is loaded, columns lowercased, and `tail_id_is` / `head_id_is`  
set to the appropriate ID namespace (UBERON_ID).

### 1.1 Monarch

In [19]:
Monarch_ANATOMY_ANATOMY = pd.read_csv(PROC_DIR + 'Monarch/Human/Hum_AnatomicalEntity_AnatomicalEntity_MONARCH.csv')
Monarch_ANATOMY_ANATOMY.columns = Monarch_ANATOMY_ANATOMY.columns.str.lower()
Monarch_ANATOMY_ANATOMY = Monarch_ANATOMY_ANATOMY.rename(columns={'tail_name':'tail_detail_name','head_name':'head_detail_name','kgsource':'kg_source'})

Monarch_ANATOMY_ANATOMY['tail_id_is'] = 'UBERON_ID'
Monarch_ANATOMY_ANATOMY['head_id_is'] = 'UBERON_ID'
Monarch_ANATOMY_ANATOMY['relation'] = 'AnatomicalEntity_AnatomicalEntity'
Monarch_ANATOMY_ANATOMY['head_type'] = 'AnatomicalEntity'
Monarch_ANATOMY_ANATOMY['tail_type'] = 'AnatomicalEntity'
Monarch_ANATOMY_ANATOMY['species'] = 'Homo sapiens'

Monarch_ANATOMY_ANATOMY['kg_type'] = 'Generalised'
Monarch_ANATOMY_ANATOMY['kg_source'] = 'Monarch_KG'

print(f"Monarch Anatomy→Anatomy: {len(Monarch_ANATOMY_ANATOMY):,} rows")
Monarch_ANATOMY_ANATOMY.head(3)

Monarch Anatomy→Anatomy: 36,890 rows


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,tail_taxon,tail_taxon_label,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,species,kg_type
0,UBERON:0000002,UBERON:0001560,subclass_of,infores:uberon,Monarch_KG,uterine cervix,AnatomicalEntity,UBERON_ID,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,AnatomicalEntity,AnatomicalEntity,AnatomicalEntity_AnatomicalEntity,Homo sapiens_Homo sapiens,Homo sapiens,Generalised
1,UBERON:0000002,UBERON:0005156,subclass_of,infores:uberon,Monarch_KG,uterine cervix,AnatomicalEntity,UBERON_ID,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,AnatomicalEntity,AnatomicalEntity,AnatomicalEntity_AnatomicalEntity,Homo sapiens_Homo sapiens,Homo sapiens,Generalised
2,UBERON:0000002,UBERON:0000995,related_to,infores:uberon,Monarch_KG,uterine cervix,AnatomicalEntity,UBERON_ID,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,AnatomicalEntity,AnatomicalEntity,AnatomicalEntity_AnatomicalEntity,Homo sapiens_Homo sapiens,Homo sapiens,Generalised


### 1.2 Pheknowlator

In [20]:
Pheknowlator_ANATOMY_ANATOMY = pd.read_csv(PROC_DIR + 'pheknowlator/Anatomy_Anatomy.csv')
Pheknowlator_ANATOMY_ANATOMY.columns = Pheknowlator_ANATOMY_ANATOMY.columns.str.lower()

Pheknowlator_ANATOMY_ANATOMY['tail_id_is'] = 'UBERON_ID'
Pheknowlator_ANATOMY_ANATOMY['head_id_is'] = 'UBERON_ID'
Pheknowlator_ANATOMY_ANATOMY['relation'] = 'AnatomicalEntity_AnatomicalEntity'
Pheknowlator_ANATOMY_ANATOMY['head_type'] = 'AnatomicalEntity'
Pheknowlator_ANATOMY_ANATOMY['tail_type'] = 'AnatomicalEntity'
Pheknowlator_ANATOMY_ANATOMY['kg_source'] = 'pheknowlator'
Pheknowlator_ANATOMY_ANATOMY['species'] = 'Homo sapiens'
Pheknowlator_ANATOMY_ANATOMY['kg_type'] = 'Generalised'
print(f"Pheknowlator Anatomy→Anatomy: {len(Pheknowlator_ANATOMY_ANATOMY):,} rows")
Pheknowlator_ANATOMY_ANATOMY.head(3)

Pheknowlator Anatomy→Anatomy: 8,489 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,species,kg_type
0,UBERON:0005043,AnatomicalEntity_AnatomicalEntity,UBERON:0010314,AnatomicalEntity,AnatomicalEntity,mucosa of nasolacrimal duct,structure with developmental contribution from...,UBERON_ID,UBERON_ID,pheknowlator,Homo sapiens,Generalised
1,UBERON:0010141,AnatomicalEntity_AnatomicalEntity,UBERON:0005295,AnatomicalEntity,AnatomicalEntity,primitive sex cord of indifferent gonad,sex cord,UBERON_ID,UBERON_ID,pheknowlator,Homo sapiens,Generalised
2,UBERON:0002349,AnatomicalEntity_AnatomicalEntity,UBERON:0018260,AnatomicalEntity,AnatomicalEntity,myocardium,layer of muscle tissue,UBERON_ID,UBERON_ID,pheknowlator,Homo sapiens,Generalised


### 1.3 PrimeKG

In [21]:
PrimeKG_ANATOMY_ANATOMY = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Anatomy_Anatomy.csv')
PrimeKG_ANATOMY_ANATOMY.columns = PrimeKG_ANATOMY_ANATOMY.columns.str.lower()

PrimeKG_ANATOMY_ANATOMY['tail_id_is'] = 'UBERON_ID'
PrimeKG_ANATOMY_ANATOMY['head_id_is'] = 'UBERON_ID'
PrimeKG_ANATOMY_ANATOMY['relation'] = 'AnatomicalEntity_AnatomicalEntity'
PrimeKG_ANATOMY_ANATOMY['head_type'] = 'AnatomicalEntity'
PrimeKG_ANATOMY_ANATOMY['tail_type'] = 'AnatomicalEntity'
PrimeKG_ANATOMY_ANATOMY['kg_source'] = 'PrimeKG'
PrimeKG_ANATOMY_ANATOMY['kg_type'] = 'Generalised'
PrimeKG_ANATOMY_ANATOMY['species'] = 'Homo sapiens'


print(f"PrimeKG Anatomy→Anatomy: {len(PrimeKG_ANATOMY_ANATOMY):,} rows")
PrimeKG_ANATOMY_ANATOMY.head(3)

PrimeKG Anatomy→Anatomy: 14,032 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0005156,AnatomicalEntity_AnatomicalEntity,UBERON:0000002,AnatomicalEntity,parent-child,AnatomicalEntity,PrimeKG,UBERON_ID,UBERON_ID,reproductive structure,uterine cervix,Generalised,Homo sapiens
1,UBERON:0000161,AnatomicalEntity_AnatomicalEntity,UBERON:0000003,AnatomicalEntity,parent-child,AnatomicalEntity,PrimeKG,UBERON_ID,UBERON_ID,orifice,naris,Generalised,Homo sapiens
2,UBERON:0010314,AnatomicalEntity_AnatomicalEntity,UBERON:0000004,AnatomicalEntity,parent-child,AnatomicalEntity,PrimeKG,UBERON_ID,UBERON_ID,structure with developmental contribution from...,nose,Generalised,Homo sapiens


### 1.4 TAR-KG

In [22]:
TARKG_ANATOMY_ANATOMY = pd.read_csv(PROC_DIR + 'TARKG/Anatomy_Anatomy.csv')
TARKG_ANATOMY_ANATOMY.columns = TARKG_ANATOMY_ANATOMY.columns.str.lower()

TARKG_ANATOMY_ANATOMY['tail_id_is'] = 'UBERON_ID'
TARKG_ANATOMY_ANATOMY['head_id_is'] = 'UBERON_ID'
TARKG_ANATOMY_ANATOMY['relation'] = 'AnatomicalEntity_AnatomicalEntity'
TARKG_ANATOMY_ANATOMY['head_type'] = 'AnatomicalEntity'
TARKG_ANATOMY_ANATOMY['tail_type'] = 'AnatomicalEntity'
TARKG_ANATOMY_ANATOMY['kg_source'] = 'TARKG'
TARKG_ANATOMY_ANATOMY['kg_type'] = 'Generalised'
TARKG_ANATOMY_ANATOMY['species'] = 'Homo sapiens'

print(f"TARKG Anatomy→Anatomy: {len(TARKG_ANATOMY_ANATOMY):,} rows")
TARKG_ANATOMY_ANATOMY.head(3)

TARKG Anatomy→Anatomy: 33,344 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,UBERON:0016446,AnatomicalEntity_AnatomicalEntity,UBERON:0001037,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,hair of head,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-23708,OpenBioLink,0,Generalised,Homo sapiens
1,UBERON:0000329,AnatomicalEntity_AnatomicalEntity,UBERON:0001037,AnatomicalEntity,PART_OF,AnatomicalEntity,TARKG,hair root,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-4216,OpenBioLink,0,Generalised,Homo sapiens
2,UBERON:0002074,AnatomicalEntity_AnatomicalEntity,UBERON:0001037,AnatomicalEntity,PART_OF,AnatomicalEntity,TARKG,hair shaft,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-7069,OpenBioLink,0,Generalised,Homo sapiens


## 2 · Consolidate into Unified Schema

All source DataFrames are aligned to `REQUIRED_COLS` (missing columns filled with `pd.NA`),  
then concatenated into `final_df`.

In [23]:
source_dfs = [
    Monarch_ANATOMY_ANATOMY,
    Pheknowlator_ANATOMY_ANATOMY,
    PrimeKG_ANATOMY_ANATOMY,
    TARKG_ANATOMY_ANATOMY
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df['species'] = 'Homo sapiens'
final_df

Consolidated rows: 92,755


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0001560,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,neck of organ,Generalised,Homo sapiens
1,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0005156,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,reproductive structure,Generalised,Homo sapiens
2,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000995,AnatomicalEntity,related_to,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,uterus,Generalised,Homo sapiens
3,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000996,AnatomicalEntity,related_to,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,vagina,Generalised,Homo sapiens
4,UBERON:0000003,AnatomicalEntity_AnatomicalEntity,UBERON:0000161,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,naris,orifice,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92750,UBERON:6005177,AnatomicalEntity_AnatomicalEntity,UBERON:6007232,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect chaeta,insect eo-type sensillum,Generalised,Homo sapiens
92751,UBERON:6007070,AnatomicalEntity_AnatomicalEntity,UBERON:6040007,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect centro-posterior medial synaptic neurop...,insect synaptic neuropil domain,Generalised,Homo sapiens
92752,UBERON:6007070,AnatomicalEntity_AnatomicalEntity,UBERON:6001925,AnatomicalEntity,PART_OF,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect centro-posterior medial synaptic neurop...,insect embryonic/larval protocerebrum,Generalised,Homo sapiens
92753,UBERON:6007232,AnatomicalEntity_AnatomicalEntity,UBERON:6007231,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect eo-type sensillum,insect external sensillum,Generalised,Homo sapiens


## 3 · Sanity Check — Distinct Values

In [24]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'AnatomicalEntity_AnatomicalEntity'}
head_type           : {'AnatomicalEntity'}
tail_type           : {'AnatomicalEntity'}
relation_type       : {'parent-child', 'IS_A', 'subclass_of', None, 'PART_OF', 'related_to'}
kg_source           : {'pheknowlator', 'PrimeKG', 'TARKG', 'Monarch_KG'}
head_id_is          : {'UBERON_ID'}
tail_id_is          : {'UBERON_ID'}
kg_type             : {'Generalised'}
species             : {'Homo sapiens'}


## 4 · NaN Audit (pre-dedup)

In [25]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,8489,0,8489
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,3818,3818,7636


In [26]:
final_df = final_df[~final_df['head_detail_name'].isna()]
final_df = final_df[~final_df['tail_detail_name'].isna()]
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0001560,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,neck of organ,Generalised,Homo sapiens
1,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0005156,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,reproductive structure,Generalised,Homo sapiens
2,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000995,AnatomicalEntity,related_to,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,uterus,Generalised,Homo sapiens
3,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000996,AnatomicalEntity,related_to,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,uterine cervix,vagina,Generalised,Homo sapiens
4,UBERON:0000003,AnatomicalEntity_AnatomicalEntity,UBERON:0000161,AnatomicalEntity,subclass_of,AnatomicalEntity,Monarch_KG,UBERON_ID,UBERON_ID,naris,orifice,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92750,UBERON:6005177,AnatomicalEntity_AnatomicalEntity,UBERON:6007232,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect chaeta,insect eo-type sensillum,Generalised,Homo sapiens
92751,UBERON:6007070,AnatomicalEntity_AnatomicalEntity,UBERON:6040007,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect centro-posterior medial synaptic neurop...,insect synaptic neuropil domain,Generalised,Homo sapiens
92752,UBERON:6007070,AnatomicalEntity_AnatomicalEntity,UBERON:6001925,AnatomicalEntity,PART_OF,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect centro-posterior medial synaptic neurop...,insect embryonic/larval protocerebrum,Generalised,Homo sapiens
92753,UBERON:6007232,AnatomicalEntity_AnatomicalEntity,UBERON:6007231,AnatomicalEntity,IS_A,AnatomicalEntity,TARKG,UBERON_ID,UBERON_ID,insect eo-type sensillum,insect external sensillum,Generalised,Homo sapiens


In [27]:
#

## 5 · Deduplication

Group on `(head, relation, tail)`. For `kg_source`, merge all unique sources with `::` delimiter.  
All other metadata columns take the first non-null value.

In [28]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
    
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 88,930  |  After dedup: 52,247


,head,relation,tail,head_type,relation_type,tail_type,head_id_is,tail_id_is,kg_source,kg_type,head_detail_name,tail_detail_name,species
0,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000995,AnatomicalEntity,related_to,AnatomicalEntity,UBERON_ID,UBERON_ID,Monarch_KG::TARKG,Generalised,uterine cervix,uterus,Homo sapiens
1,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0000996,AnatomicalEntity,related_to,AnatomicalEntity,UBERON_ID,UBERON_ID,Monarch_KG,Generalised,uterine cervix,vagina,Homo sapiens
2,UBERON:0000002,AnatomicalEntity_AnatomicalEntity,UBERON:0001560,AnatomicalEntity,subclass_of,AnatomicalEntity,UBERON_ID,UBERON_ID,Monarch_KG::TARKG::pheknowlator,Generalised,uterine cervix,neck of organ,Homo sapiens


## 6 · Post-dedup NaN Audit & Source Distribution

In [29]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

display(pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
}))

print("\nkg_source values present:", set(final_df_dedup['kg_source']))

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,74,0,74
tail_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
kg_source,0,0,0
kg_type,0,0,0



kg_source values present: {'Monarch_KG::TARKG', 'PrimeKG', 'Monarch_KG', 'pheknowlator', 'TARKG::pheknowlator', 'TARKG', 'Monarch_KG::TARKG::pheknowlator', 'Monarch_KG::pheknowlator', 'Monarch_KG::PrimeKG'}


In [30]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df_dedup[col])}")

relation            : {'AnatomicalEntity_AnatomicalEntity'}
head_type           : {'AnatomicalEntity'}
tail_type           : {'AnatomicalEntity'}
relation_type       : {'parent-child', 'IS_A', 'subclass_of', None, 'PART_OF', 'related_to'}
kg_source           : {'Monarch_KG::TARKG', 'PrimeKG', 'Monarch_KG', 'pheknowlator', 'TARKG::pheknowlator', 'TARKG', 'Monarch_KG::TARKG::pheknowlator', 'Monarch_KG::pheknowlator', 'Monarch_KG::PrimeKG'}
head_id_is          : {'UBERON_ID'}
tail_id_is          : {'UBERON_ID'}
kg_type             : {'Generalised'}
species             : {'Homo sapiens'}


## 7 · Save Output

In [31]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


In [32]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 52,247 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/ANATOMY_ANATOMY/ALL_ANATOMY_ANATOMY.csv
